In [3]:
import re
import numpy as np
import pandas as pd

In [4]:
PATH = "/Users/parin/Desktop/prosecutor explorer/worcester/Worcester DAMION data.xlsx"   # adjust if needed

# see what sheets are in it
xl = pd.ExcelFile(PATH, engine="openpyxl")
print(xl.sheet_names)

# load every sheet into a dict of DataFrames
sheets = {name: xl.parse(name) for name in xl.sheet_names}
for name, d in sheets.items():
    print(f"\n{name}: {d.shape[0]:,} rows x {d.shape[1]} cols")
    print(d.columns.tolist())

# if it's a single sheet, or you want the first one:
worcester_raw = sheets[xl.sheet_names[0]]
worcester_raw.head()

['ACLU_2014_to_2019']

ACLU_2014_to_2019: 337,067 rows x 23 cols
['Case ID number', 'Defendant ID number', 'Offense date', 'Case filing date', 'PRSCTR_OFC', 'Court name', 'Criminal count number', 'Charge code', 'Charge description', 'Charge type', 'Police department', 'Charge initiation', 'Defendants race/ethnicity', 'Defendants gender', 'Judge name', 'Disposition date', 'Disposition code', 'Disposition description', 'Disposition type', 'Prosecutor disposition', 'Sentence type', 'Sentence description', 'Case status']


,Case ID number,Defendant ID number,Offense date,Case filing date,PRSCTR_OFC,Court name,Criminal count number,Charge code,Charge description,Charge type,...,Defendants gender,Judge name,Disposition date,Disposition code,Disposition description,Disposition type,Prosecutor disposition,Sentence type,Sentence description,Case status
0,14-01-000080-1,8810,2013-06-17,2014-01-02,WOSC,Worcester Superior Court,1,265/18(b),ARMED ASSAULT W/INT. TO MURDER,Intent to Murder,...,M,"Kenton-Walker, Janet",2014-08-14,GPA,Guilty,NaN,NaN,NaN,"Cts. 1, 2; GP 8 - 12 yrs. MCI Cedar Jct.\nCts....",CLSD
1,14-01-000080-1,8810,2013-06-17,2014-01-02,WOSC,Worcester Superior Court,2,265/18/F,"ASSAULT TO ROB, ARMED c265 §18(b)",Robbery Armed,...,M,"Kenton-Walker, Janet",2014-08-14,GPA,Guilty,NaN,NaN,NaN,NaN,CLSD
2,14-01-000080-1,8810,2013-06-17,2014-01-02,WOSC,Worcester Superior Court,3,265/18/F,"ASSAULT TO ROB, ARMED c265 §18(b)",Robbery Armed,...,M,"Kenton-Walker, Janet",2014-08-14,DISM,Dismissed,NaN,NaN,NaN,NaN,CLSD
3,14-01-000080-1,8810,2013-06-17,2014-01-02,WOSC,Worcester Superior Court,4,269/10,POSS. OF FIREARM NOT HOME/WORK,Firearms General,...,M,"Kenton-Walker, Janet",2014-08-14,GPA,Guilty,NaN,NaN,NaN,NaN,CLSD
4,14-01-000080-1,8810,2013-06-17,2014-01-02,WOSC,Worcester Superior Court,5,269/10/TT,"AMMUNITION W/O FID CARD, POSSESS c269s10(h)(1)",Firearms Ammunition,...,M,"Kenton-Walker, Janet",2014-08-14,GPA,Guilty,NaN,NaN,NaN,NaN,CLSD


In [10]:
YEARS = (2017, 2024)
MISSING_MAX = 0.20
MIN_CELL = 30
POP = {y: 862_111 for y in range(2017, 2025)}   # 2020 Census; swap in annual PEP/ACS
EVENTS = {"MA CJ reform act": "2018-04-13", "COVID court closures": "2020-03-16"}
DEF_COL = "def_filled"

RENAME = {
    "Case ID number": "case_id", "Defendant ID number": "defendant_id",
    "Offense date": "offense_date", "Case filing date": "filing_date",
    "Court name": "court_name", "Criminal count number": "charge_count_number",
    "Charge code": "charge_code", "Charge description": "charge_description",
    "Charge type": "charge_type", "Police department": "filing_agency",
    "Charge initiation": "charge_initiation",
    "Defendants race/ethnicity": "defendant_race", "Defendants gender": "defendant_sex",
    "Judge name": "judge_name", "Disposition date": "disposition_date",
    "Disposition code": "disposition_code", "Disposition description": "disposition_description",
    "Disposition type": "disposition_type", "Prosecutor disposition": "prosecutor_disposition",
    "Sentence type": "sentence_type", "Sentence description": "sentence_description",
    "Case status": "case_status",
}

def load_w(d):
    d = d.rename(columns=RENAME).copy()
    print("missing std cols:", [c for c in RENAME.values() if c not in d])
    for col in ("charge_description", "disposition_description", "sentence_description"):
        d[col] = (d[col].astype("string").str.replace("Â§", "§", regex=False)
                  .str.replace("Â—", "-", regex=False).str.replace("Â", "", regex=False).str.strip())
    for col in ("offense_date", "filing_date", "disposition_date"):
        if d[col].dtype == object:
            d[col] = pd.to_datetime(d[col], format="%d-%b-%y", errors="coerce")
        else:
            d[col] = pd.to_datetime(d[col], errors="coerce")
    bad = (d.disposition_date < "1986-01-01") | (d.disposition_date > pd.Timestamp.today())
    d.loc[bad, "disposition_date"] = pd.NaT
    n = len(d); d = d.drop_duplicates(); print(f"dropped {n - len(d):,} dup rows")
    g = d.groupby("case_id").defendant_id
    n_ids = g.nunique()
    d["def_filled"] = d.defendant_id.fillna(d.case_id.map(g.first().where(n_ids.eq(1))))
    d["case_key"] = d.case_id.astype(str) + "|" + d.def_filled.astype(str)
    print("cases with 2+ defendants:", int((n_ids > 1).sum()))
    print("filing date range:", d.filing_date.min(), "to", d.filing_date.max())
    print("disposition date null %:", round(100 * d.disposition_date.isna().mean(), 1))
    return d

df = load_w(worcester_raw)

def pct(x): return (100 * x).round(1)
def share(s):
    t = s.value_counts(dropna=False)
    return pd.DataFrame({"n": t, "pct": pct(t / t.sum())})
def rate_by(frame, flag, by):
    t = frame.assign(f=np.asarray(flag)).groupby(by, observed=True).f.agg(n="size", k="sum")
    t["pct"] = pct(t.k / t.n)
    return t

missing std cols: []
dropped 13 dup rows
cases with 2+ defendants: 1
filing date range: 2014-01-02 00:00:00 to 2019-12-31 00:00:00
disposition date null %: 0.1


In [11]:
for col in ["disposition_code", "disposition_type", "prosecutor_disposition", "charge_initiation",
            "charge_type", "sentence_type", "case_status", "defendant_race", "defendant_sex"]:
    print(f"\n{col}  ({pct(df[col].isna().mean())}% null)")
    print(df[col].value_counts(dropna=False).head(25).to_string())
print("\nsentence_description sample:\n", df.sentence_description.dropna().sample(5, random_state=1).to_string())


disposition_code  (0.0% null)
disposition_code
DISM    133653
NRSP     41711
GPA      39705
CWOF     34463
RFN      21711
RFL      16142
GFL      10405
DWOP      7763
PTP       6539
RSP       5771
GFN       5731
JWNG      5107
JTNG      2654
NOL       1865
JTG       1011
DPCC       699
JWG        577
TR         567
SGL        399
FWOP       131
RCH        126
DIRV       107
DWP         52
RFNG        45
AS          38

disposition_type  (100.0% null)
disposition_type
NaN    337054

prosecutor_disposition  (100.0% null)
prosecutor_disposition
NaN    337054

charge_initiation  (100.0% null)
charge_initiation
NaN    337054

charge_type  (0.6% null)
charge_type
MV Moving Violation                 78215
MV License/RMV/Reporting            33566
MV Other                            25021
All Other - General                 17755
Larceny                             17539
Assault & Battery                   13284
OUI Liquor                          11021
Domestic Violence Assault           109

In [12]:
W_DISPO = {
    "DISM": "dismissal", "DWOP": "dismissal", "DWP": "dismissal", "NOL": "dismissal",
    "PCNF": "dismissal", "AS": "dismissal",
    "PTP": "diversion", "DPCC": "diversion", "CATI": "diversion", "VAA": "diversion",
    "CWOF": "cwof", "GPCW": "cwof",
    "GPA": "plea", "GFL": "plea", "GFN": "plea", "SGL": "plea", "GYO": "plea",   # "Guilty" codes assumed plea
    "DPA": "plea", "DFL": "plea", "DFN": "plea",
    "JTG": "trial_conviction", "JWG": "trial_conviction", "SJTL": "trial_conviction",
    "JWNG": "acquittal", "JTNG": "acquittal", "DIRV": "acquittal", "RFNG": "acquittal",
    "NGYO": "acquittal", "NCR": "acquittal",
    "NRSP": "civil", "RFN": "civil", "RFL": "civil", "RSP": "civil",
    "TR": "other", "RCH": "other", "FWOP": "other", "NCF": "other", "PCF": "other",
    "MT": "other", "DPCW": "other",
}
W_LESSER = {"SGL", "SJTL"}

CHAPTER_CAT = {"90": "Motor Vehicle", "89": "Motor Vehicle", "90B": "Motor Vehicle", "90C": "Motor Vehicle",
               "265": "Person", "266": "Property", "94C": "Drug",
               "209A": "Protective order", "258E": "Protective order",
               "269": "Firearms/Weapons", "140": "Firearms/Weapons", "138": "Alcohol",
               "272": "Disorderly/Public Order", "275": "Disorderly/Public Order",
               "268": "Obstruction/justice", "268A": "Obstruction/justice"}

def offense_category(code):
    m = re.match(r"^C?(\d+[A-Z]?)(?=[/S*(]|$)", str(code).strip().upper())
    return CHAPTER_CAT.get(m.group(1), "Other/Unknown") if m else "Other/Unknown"

RACE_MAP = {"American Indian": "Other", "Pacific Islander": "Other", "Middle Eastern": "Other"}
SEX_MAP = {"M": "Male", "F": "Female", "Male": "Male", "Female": "Female"}

df["outcome"] = df.disposition_code.map(W_DISPO)
df.loc[df.disposition_code.isna(), "outcome"] = "pending"
print("unmapped codes:", df.loc[df.outcome.isna(), "disposition_code"].value_counts().to_dict())
df["outcome"] = df.outcome.fillna("other")
df["lesser"] = df.disposition_code.isin(W_LESSER)

code = df.charge_code.astype(str)
df["placeholder"] = code.isin(["666666", "999999"])
lut = {c: offense_category(c) for c in code.unique()}
df["offense_cat"] = np.where(df.placeholder, "Other/Unknown", code.map(lut))

df["is_civil"] = df.charge_description.str.contains(r"\s\*(\s|$)", na=False) | df.outcome.eq("civil")
superior = df.court_name.astype(str).str.contains("Superior", case=False, na=False)
df["sev_rank"] = np.where(df.is_civil, 0, np.where(superior, 3, 1))     # no statutory class in raw data
df["sev_class"] = np.select([df.is_civil, superior], ["Civil", "Superior Ct"], "District Ct")

df["race"] = df.defendant_race.replace(RACE_MAP).fillna("Unknown").replace({"": "Unknown", "None": "Unknown"})
df["sex"] = df.defendant_sex.map(SEX_MAP).fillna("Unknown")
df["pend"] = df.outcome.eq("pending")
print(df.race.value_counts().to_string())

unmapped codes: {}


/var/folders/w6/qdlw4d216zx010j_cs5_j5fc0000gn/T/ipykernel_44881/3208736839.py:42: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["is_civil"] = df.charge_description.str.contains(r"\s\*(\s|$)", na=False) | df.outcome.eq("civil")


race
White                               207068
Hispanic/Latino                      59918
Black or African American            39125
Unknown                              21971
Caucasion                             4284
Asian                                 3812
East Indian                            530
Other                                  130
Arabic                                 101
Hawaiian Native/Pacific Islander        71
Multiracial or Other                    25
African American                        19


In [13]:
OUTCOME_RANK = {"trial_conviction": 8, "plea": 7, "cwof": 6, "acquittal": 5, "diversion": 4,
                "dismissal": 3, "declined": 2, "other": 1, "pending": 0}

lead = (df.sort_values(["case_key", "sev_rank"], ascending=[True, False])
          .drop_duplicates("case_key").set_index("case_key")
          [[DEF_COL, "sev_rank", "sev_class", "offense_cat", "race", "sex", "charge_description", "court_name"]])
agg = df.groupby("case_key").agg(filed=("filing_date", "min"), dispo=("disposition_date", "max"),
                                 n_charges=("case_key", "size"),
                                 all_civil=("is_civil", "all"), any_pending=("pend", "any"))
cr = df[~df.is_civil]
outcome = (cr.assign(r=cr.outcome.map(OUTCOME_RANK)).sort_values("r", ascending=False)
             .drop_duplicates("case_key").set_index("case_key").outcome)

cases_all = lead.join(agg).join(outcome)
cases_all["outcome"] = cases_all.outcome.fillna("civil")
cases_all["any_lesser_plea"] = cases_all.index.isin(cr.loc[cr.outcome.eq("plea") & cr.lesser, "case_key"].unique())
cases_all["year"] = cases_all.filed.dt.year
dd = (cases_all.dispo - cases_all.filed).dt.days
cases_all["days_to_dispo"] = dd.where((dd >= 0) & ~cases_all.any_pending)

p = cases_all[~cases_all.all_civil & cases_all[DEF_COL].notna()].sort_values("filed")
cases_all["prior_cases"] = p.groupby(DEF_COL).cumcount().reindex(cases_all.index)
cases_all["prior_level"] = (pd.cut(cases_all.prior_cases, [-1, 0, 2, np.inf], labels=["0", "1-2", "3+"])
                              .astype(object).fillna("Unknown"))

cases = cases_all[cases_all.year.between(*YEARS)]
ch_w = df[df.filing_date.dt.year.between(*YEARS)]
crim = cases[~cases.all_civil & cases.outcome.ne("pending")]
charged = crim[crim.outcome.ne("declined")]
convicted = charged[charged.outcome.isin(["plea", "trial_conviction"])]
print(len(cases), "cases |", len(crim), "criminal disposed |", len(charged), "charged |", len(convicted), "convicted")

73405 cases | 64236 criminal disposed | 64236 charged | 14915 convicted


In [14]:
def sent_class(txt):
    if not isinstance(txt, str) or not txt.strip():
        return None
    t = txt.lower()
    if "susp" in t: return "Probation (suspended)"
    if re.search(r"mci|hoc|house of corr|jail|prison|committ|dys|\bcom\b", t): return "Incarceration"
    if "prob" in t: return "Probation"
    if re.search(r"fine|cost|restitution|community serv", t): return "Fine/other"
    if "filed" in t: return "Filed (no sentence)"
    return None

CT_LINE = re.compile(r"^\s*cts?\.?\s*([\d,\s&\-and]+);?(.*)$", re.I)
def split_counts(text):
    by_ct, default = {}, None
    for line in str(text).splitlines():
        m = CT_LINE.match(line)
        if not m:
            default = (default or "") + " " + line
            continue
        nums = set()
        for a, b in re.findall(r"(\d+)\s*-\s*(\d+)", m.group(1)):
            nums.update(range(int(a), int(b) + 1))
        nums.update(int(n) for n in re.findall(r"\d+", re.sub(r"\d+\s*-\s*\d+", "", m.group(1))))
        for n in nums:
            by_ct[n] = m.group(2)
    return by_ct, default

conv_all = df[df.outcome.isin(["plea", "trial_conviction"]) & ~df.is_civil].copy()
cache = {t: split_counts(t) for t in conv_all.sentence_description.dropna().unique()}
def seg(row):
    t = row.sentence_description
    if pd.isna(t): return None
    by_ct, default = cache[t]
    ct = pd.to_numeric(row.charge_count_number, errors="coerce")
    return by_ct.get(int(ct), default) if pd.notna(ct) else default
conv_all["sent_seg"] = conv_all.apply(seg, axis=1)

conv_all["sent"] = conv_all.sentence_type.astype("string").map(sent_class)
conv_all["sent"] = conv_all.sent.fillna(conv_all.sent_seg.map(sent_class))
conv_all.loc[conv_all.sent.isna() & conv_all.disposition_code.eq("GFL"), "sent"] = "Filed (no sentence)"
conv_all.loc[conv_all.sent.isna() & conv_all.disposition_code.eq("GFN"), "sent"] = "Fine/other"
conv_all["sent"] = conv_all.sent.fillna("Missing")

UNIT = {"y": 365.25, "m": 30.44, "d": 1, "w": 7}
TERM = re.compile(r"(\d+(?:\.\d+)?)\s*(?:-|to)?\s*(\d+(?:\.\d+)?)?\s*(yrs?|years?|mos?|months?|days?|wks?|weeks?)\b", re.I)
def term_days(s):
    if not isinstance(s, str) or "susp" in s.lower():
        return (np.nan, np.nan)
    m = TERM.search(s)
    if not m:
        return (np.nan, np.nan)
    u = UNIT[m.group(3)[0].lower()]
    lo = float(m.group(1)); hi = float(m.group(2)) if m.group(2) else lo
    return (lo * u, hi * u)
terms = conv_all.sent_seg.map(term_days)
conv_all["term_min_days"] = terms.str[0]
conv_all["term_max_days"] = terms.str[1]

sent_missing = conv_all.loc[conv_all.filing_date.dt.year.between(*YEARS), "sent"].eq("Missing").mean()
HAS_INCARC = sent_missing <= MISSING_MAX
print(f"sentence missing among convictions: {100 * sent_missing:.1f}% -> HAS_INCARC={HAS_INCARC}")
print(conv_all[["sentence_type", "sentence_description", "charge_count_number", "sent_seg", "sent",
                "term_min_days", "term_max_days"]]
      .dropna(subset=["sentence_description"]).sample(10, random_state=2).to_string())   # eyeball the parser

rank = {"Incarceration": 6, "Probation (suspended)": 5, "Probation": 5, "Fine/other": 4,
        "Filed (no sentence)": 2, "Missing": 0}
case_sent = (conv_all.assign(r=conv_all.sent.map(rank)).sort_values("r", ascending=False)
                     .drop_duplicates("case_key").set_index("case_key").sent)

sentence missing among convictions: 17.3% -> HAS_INCARC=True
        sentence_type                                                                                              sentence_description  charge_count_number                                                                                                           sent_seg                   sent  term_min_days  term_max_days
257297            NaN                                                                                 90 DAYS CREDIT FOR 90 DAYS SERVED                    3                                                                                  90 DAYS CREDIT FOR 90 DAYS SERVED                Missing         90.000         90.000
267333            NaN                                                                                                          3 MONTHS                    1                                                                                                           3 MONTHS    Filed (no sent

In [15]:
yr = ch_w.filing_date.dt.year
cv_w = conv_all[conv_all.filing_date.dt.year.between(*YEARS)]
inc_w = cv_w[cv_w.sent.eq("Incarceration")]
groups = [
    ({"defendant_id": ch_w[DEF_COL].isna(), "filing_date": ch_w.filing_date.isna(),
      "disposition_date": ch_w.disposition_date.isna(),
      "charge_code": ch_w.charge_code.isna() | ch_w.placeholder,
      "disposition_code": ch_w.disposition_code.isna(),
      "race": ch_w.race.eq("Unknown"), "sex": ch_w.sex.eq("Unknown"),
      "court": ch_w.court_name.isna()}, yr),
    ({"sentence_type (convictions)": cv_w.sent.eq("Missing")}, cv_w.filing_date.dt.year),
    ({"sentence_length (incarcerated)": inc_w.term_max_days.isna()}, inc_w.filing_date.dt.year),
]
rows = []
for checks, yrs in groups:
    for k, miss in checks.items():
        by_year = (~miss).groupby(yrs).mean()
        rows.append(dict(field=k, pct_missing=pct(miss.mean()),
                         years_available=",".join(str(int(y)) for y, v in by_year.items() if v > 0),
                         status="ok" if miss.mean() <= MISSING_MAX else "insufficient data"))
rows += [dict(field="severity (felony/misd)", pct_missing=100.0, years_available="",
              status="insufficient data (court-level proxy only)"),
         dict(field="dob/age", pct_missing=100.0, years_available="", status="insufficient data (not delivered)"),
         dict(field="bail/detention", pct_missing=100.0, years_available="", status="insufficient data (not delivered)")]
coverage = pd.DataFrame(rows)
coverage

,field,pct_missing,years_available,status
0,defendant_id,0.0,"2017,2018,2019",ok
1,filing_date,0.0,"2017,2018,2019",ok
2,disposition_date,0.2,"2017,2018,2019",ok
3,charge_code,0.7,"2017,2018,2019",ok
4,disposition_code,0.0,"2017,2018,2019",ok
5,race,6.0,"2017,2018,2019",ok
6,sex,0.8,"2017,2018,2019",ok
7,court,0.0,"2017,2018,2019",ok
8,sentence_type (convictions),17.3,"2017,2018,2019",ok
9,sentence_length (incarcerated),1.3,"2017,2018,2019",ok


## Q1 - Q2

In [16]:
vol = cases[~cases.all_civil].groupby("year").size().rename("cases").to_frame()
vol["civil_only_excluded"] = cases[cases.all_civil].groupby("year").size()
vol["yoy_pct"] = pct(vol.cases.pct_change())
vol["vs_2017_pct"] = pct(vol.cases / vol.cases.iloc[0] - 1)
vol["per_10k"] = (vol.cases / vol.index.map(POP) * 1e4).round(1)
vol

,cases,civil_only_excluded,yoy_pct,vs_2017_pct,per_10k
year,,,,,
2017,23685,2971,NaN,0.0,274.7
2018,22656,3112,-4.3,-4.3,262.8
2019,17895,3086,-21.0,-24.4,207.6


In [17]:
crim_all = cases[~cases.all_civil]
q3_court = share(crim_all.sev_class)        # proxy only; felony/misd = insufficient data
q3_cat = share(crim_all.offense_cat).head(5)
q3_charges = share(crim_all.charge_description.str.replace(r"\s+c\d+[A-Z]?\s*§.*$", "", regex=True)).head(5)
q3_court, q3_cat, q3_charges

(                 n   pct
 sev_class               
 District Ct  63673  99.1
 Superior Ct    563   0.9,
                              n   pct
 offense_cat                         
 Motor Vehicle            30139  46.9
 Person                   12087  18.8
 Property                  9278  14.4
 Drug                      4558   7.1
 Disorderly/Public Order   2195   3.4,
                                    n   pct
 charge_description                        
 LICENSE SUSPENDED, OP MV WITH   9030  14.1
 UNLICENSED OPERATION OF MV      4339   6.8
 A&B ON FAMILY/HOUSEHOLD MEMBER  4318   6.7
 OUI?LIQUOR OR .08%              3499   5.4
 UNINSURED MOTOR VEHICLE         2926   4.6)